# OpenBind-HIPPO

- **Target: A71EV2A**
- **Cycle: 01**

## Imports

In [2]:
%load_ext autoreload
%autoreload 2
import hippo
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil
import molparse as mp
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## Config

In [3]:
target_name = "A71EV2A"
target_dir = Path(environ["BULK"]) / "TARGETS" / target_name
cycle_name = "cycle_01"
cycle_dir = Path(cycle_name)
aligned_dir = target_dir / "aligned_files"

## Animal

In [4]:
animal = hippo.HIPPO(target_name, target_dir / f"{target_name}.sqlite")

 Creating HIPPO animal

name = A71EV2A

db_path = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/A71EV2A/A71EV2A.sqlite

DEBUG: hippo.Database.__init__()

DEBUG: Database.path = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/A71EV2A/A71EV2A.sqlite

DEBUG: hippo.Database.connect()

DEBUG: sqlite3.version='2.6.0'

 Success  Database connected @ /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/A71EV2A/A71EV2A.sqlite!

 Success  Initialised animal HIPPO("A71EV2A")!

## Queue BulkDock (re)placements

In [ ]:
scaffold_poses = animal.poses(tag="openbind_a71ev2a_c1_scaffolds_chemok_bbok")
elaborated_scaffolds = scaffold_poses.compounds.elabs.scaffolds
elaborated_scaffold_poses = animal.poses[set(elaborated_scaffolds.poses.ids).intersection(set(scaffold_poses.ids))]
elaborated_scaffold_poses

In [ ]:
data = []
for i,pose in enumerate(elaborated_scaffold_poses):
    print(i, pose)
    compound = pose.compound
    elabs = compound.elabs
    reference = pose.reference
    inspirations = pose.inspirations

    inspiration_d = dict()
    for i, name in enumerate(inspirations.names):
        inspiration_d[f"hit{i+1}"] = name
    
    for elab in mrich.track(elabs):
        d = dict(smiles=elab.smiles)
        d.update(inspiration_d)
        data.append(d)

df = pd.DataFrame(data)
df.to_csv("cycle_01/syndirella/elabs/a71ev2a_c1_elab_bulkdock_input.csv", index=False)
df

## Define chemspace

In [32]:
scaffolds = animal.compounds(tag="openbind_a71ev2a_c1_scaffolds_chemok_bbok")
scaffolds

compounds tagged openbind_a71ev2a_c1_scaffolds_chemok_bbok: {C × 558}

In [33]:
elabs = scaffolds.elabs
elabs

{C × 18456}

In [34]:
elaborated_scaffolds = elabs.scaffolds
elaborated_scaffolds

{C × 171}

In [35]:
# elab_poses = elabs.poses.filter(key="distance_score", value="2.0", operator="<=")
# elab_poses = elab_poses.filter(key="energy_score", value="0.0", operator="<=")
elab_poses = elabs.poses
elab_poses

{P × 36827}

In [36]:
posed_elabs = elab_poses.compounds
posed_elabs

{C × 17070}

In [37]:
posed_elabs.add_tag("openbind_a71ev2a_c1_elabs")

Tagged {C × 17070} w/ "openbind_a71ev2a_c1_elabs"

In [38]:
posed_elabs.scaffolds.add_tag("openbind_a71ev2a_c1_elaborated_scaffolds")

SQLite Database was locked, retrying...

Tagged {C × 164} w/ "openbind_a71ev2a_c1_elaborated_scaffolds"

In [39]:
chemspace = posed_elabs + posed_elabs.scaffolds
chemspace

{C × 17234}

In [40]:
filtered_elab_poses = elabs.poses.filter(key="distance_score", value="2.0", operator="<=").filter(key="energy_score", value="0.0", operator="<=")
filtered_elab_poses.add_tag("openbind_a71ev2a_c1_elabs_filtered")
filtered_posed_elabs = filtered_elab_poses.compounds
filtered_posed_elabs.add_tag("openbind_a71ev2a_c1_elabs_filtered")
chemspace_filtered = filtered_posed_elabs + posed_elabs.scaffolds
chemspace_filtered

Tagged {P × 5036} w/ "openbind_a71ev2a_c1_elabs_filtered"

Tagged {C × 1965} w/ "openbind_a71ev2a_c1_elabs_filtered"

{C × 2129}

In [ ]:
pass_ids = set()
fail_ids = set()
for pose in mrich.track(animal.poses, total=animal.num_poses):
    mrich.set_progress_field("pass", len(pass_ids))
    try:
        ok = pose.posebusters()
    except Exception as e:
        mrich.error(pose, e)
        ok = False
        
    if ok:
        pass_ids.add(pose.id)
    else:
        fail_ids.add(pose.id)

## Export chemspace

In [18]:
chemspace_poses = chemspace.best_placed_poses
chemspace_poses

{P × 17234}

In [ ]:
%%time
chemspace_poses.to_fragalysis("cycle_01/syndirella/elabs/openbind_a71ev2a_c1_elab_chemspace.sdf",
    method="openbind_a71ev2a_c1_elab_chemspace", 
    submitter_name="Max Winokan", 
    submitter_institution="DLS", 
    submitter_email="max.winokan@diamond.ac.uk",
    copy_reference_pdbs=True,
    metadata=False,
    tags=False,
    subsites=False,
)

In [19]:
chemspace_filtered_poses = filtered_elab_poses.get_best_placed_poses_per_compound() + posed_elabs.scaffolds.best_placed_poses
chemspace_filtered_poses

{P × 2129}

In [ ]:
%%time
chemspace_filtered_poses.to_fragalysis("cycle_01/syndirella/elabs/openbind_a71ev2a_c1_elab_chemspace_filtered.sdf",
    method="openbind_a71ev2a_c1_elab_chemspace_filtered", 
    submitter_name="Max Winokan", 
    submitter_institution="DLS", 
    submitter_email="max.winokan@diamond.ac.uk",
    copy_reference_pdbs=True,
    metadata=False,
    tags=False,
    subsites=False,
)

In [ ]:
chemspace_filtered_poses[7].render()

## Full chemspace recipe

In [4]:
posed_chemspace = animal.compounds(tag="openbind_a71ev2a_c1_elabs_filtered")
posed_chemspace

compounds tagged openbind_a71ev2a_c1_elabs_filtered: {C × 1942}

In [5]:
recipe = hippo.Recipe.from_compounds(
    posed_chemspace,
    pick_first=True,
    pick_cheapest=False,
    quoted_only=True,
    use_routes=True
)
recipe

#compounds = 1942

Output()

Solving recipe combinations...

Output()

Recipe({Ingredient × 1467} --> {Ingredient × 1942} via {R × 1942})

In [6]:
recipe.write_json("cycle_01/rgen/openbind_a71ev2a_c1_elabs_filtered_recipe.json", price=False)

 DISK  Writing 
/opt/xchem-fragalysis-2/maxwin/openbind-hippo/a71ev2a/cycle_01/rgen/openbind_a71ev2a_c1_elabs_filtered_recipe.json.
..

ERROR! Session/line number was not unique in database. History logging moved to new session 935


In [7]:
recipe.write_reactant_csv("cycle_01/rgen/openbind_a71ev2a_c1_elabs_filtered_reactants.csv")

DEBUG: Querying database for routes

DEBUG: Assembling route dictionary

DEBUG: Checking availability

 DISK  Writing cycle_01/rgen/openbind_a71ev2a_c1_elabs_filtered_reactants.csv...

## Add in-stock reactants

In [ ]:
df = pd.read_csv("cycle_01/rgen/a71ev2a_c1_scaffolds_recipes/Recipe_1GB7RAD_reactants.csv")
df

In [ ]:
for i,row in df.iterrows():
    compound = animal.compounds[row["compound_id"]]
    quote = compound.add_stock(
        amount=row["quoted_amount_mg"],
        purity=row["quoted_purity"],
        entry=row["quote_entry"],
    )
    print(compound, quote)
    # break

In [ ]:
animal.C283111.get_quotes(df=True)

## Add combined BB quote

In [5]:
pd.read_excel("../combined/Q2135669_2025-11(nov)-14-OpenBind-buildingblocks-c1(chempspace).xlsx").head()

,quote_entry,quoted_smiles,HIPPO Compound ID (xx01zvns2b),HIPPO Compound ID (a71ev2a),HIPPO Compound ID (d68ev3c),quote-1mg,quote-10mg
0,EN300-106689,Cl.Cl.Nc1ccc2c(c1)CNC2,61484.0,NaN,NaN,19,30
1,EN300-176585,O=C(O)c1cc(F)cc2[nH]nnc12,123049.0,NaN,NaN,30,48
2,EN300-1590558,N#Cc1c[nH]c2c(C(=O)O)cccc12,122834.0,NaN,NaN,14,19
3,EN300-26677505,CC(CN)c1ccncc1.Cl.Cl,618.0,NaN,NaN,33,55
4,EN300-6764593,COCc1ccc(Br)nc1,129215.0,NaN,NaN,14,24


In [5]:
# clear existing quotes
animal.db.backup()
animal.db.execute("DELETE FROM quote")
animal.db.commit()

test

 DISK  Writing /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/A71EV2A/A71EV2A_1QNHVE8.sqlite...

DEBUG: Copied 10000 of 877009 pages...

DEBUG: Copied 20000 of 877009 pages...

DEBUG: Copied 30000 of 877009 pages...

...

DEBUG: Copied 840000 of 877009 pages...

DEBUG: Copied 850000 of 877009 pages...

DEBUG: Copied 860000 of 877009 pages...

DEBUG: Copied 870000 of 877009 pages...

DEBUG: Copied 877009 of 877009 pages...

In [6]:
q = animal.add_enamine_quote(
    "../combined/Q2135669_2025-11(nov)-14-OpenBind-buildingblocks-c1(chempspace).xlsx",
    orig_name_col = "HIPPO Compound ID (a71ev2a)",
    smiles_col = "quoted_smiles",
    entry_col = "quote_entry",
    fixed_amount = 1,
    price_col = "quote-1mg",
    fixed_purity = None,
    fixed_lead_time = None,
    dry_run = False,
    currency = "EUR",
    allow_no_catalogue_col = True,
    orig_name_is_hippo_id=True,
    warn_nan_orig_name = False,
    overwrite_existing_quotes=False,
    delete_unavailable = False,
    debug = False,
)
q

Output()

 Warning  SMILES changed during compound registration: c1cc(C2CCNC2)[nH]n1 --> c1cc(C2CCNC2)n[nH]1!

{Ingredient × 293}

In [7]:
q = animal.add_enamine_quote(
    "../combined/Q2135669_2025-11(nov)-14-OpenBind-buildingblocks-c1(chempspace).xlsx",
    orig_name_col = "HIPPO Compound ID (a71ev2a)",
    smiles_col = "quoted_smiles",
    entry_col = "quote_entry",
    fixed_amount = 10,
    price_col = "quote-10mg",
    fixed_purity = None,
    fixed_lead_time = None,
    dry_run = False,
    currency = "EUR",
    allow_no_catalogue_col = True,
    orig_name_is_hippo_id=True,
    warn_nan_orig_name = False,
    overwrite_existing_quotes=False,
    delete_unavailable = False,
    debug = False,
)
q

Output()

 Warning  SMILES changed during compound registration: c1cc(C2CCNC2)[nH]n1 --> c1cc(C2CCNC2)n[nH]1!

{Ingredient × 293}

## Rgen

In [41]:
scaffold_chemsample_recipe = hippo.Recipe.from_json(animal.db, "cycle_01/rgen/a71ev2a_c1_scaffolds_recipes/Recipe_1GB7RAD.json")

 DISK  Reading cycle_01/rgen/a71ev2a_c1_scaffolds_recipes/Recipe_1GB7RAD.json...

Recipe was generated at: 2025-10-08 11:18:27.986227

reactants = {Ingredient × 182}

intermediates = {Ingredient × 0}

products = {Ingredient × 99}

reactions = {R × 99}

compounds = {Ingredient × 0}

In [42]:
# subtract planned synthesis cycle
product_pool = chemspace_filtered - scaffold_chemsample_recipe.product_compounds
product_pool

{C × 2106}

In [43]:
route_pool = hippo.RouteSet.from_product_ids(animal.db, product_pool.ids)
route_pool

Output()

{Route × 2102}

In [44]:
route_pool = route_pool.prune_unavailable(["Enamine", "Stock"])
route_pool

#routes before pruning = 2102

#routes after pruning = 2102

Output()

{Route × 2102}

### Start with cheapest analogues using existing reactants?

In [45]:
route = list(route_pool.routes)[0]
route, route.price

(Route #6: C141599, €582.10 EUR)

In [47]:
data = []
for route in mrich.track(route_pool):
    data.append(dict(
        product_id=route.product.id, 
        route_id=route.id, 
        # chemistry=route.reactions.types[0], 
        price=route.price.amount
    ))
df = pd.DataFrame(data)
df

Output()

,product_id,route_id,price
0,141599,6,582.1
1,204781,42,29.0
2,266892,76,18.0
3,195109,90,56.6
4,114736,187,214.0
...,...,...,...
2097,315379,20357,1025.0
2098,315384,20362,764.0
2099,315385,20363,446.0
2100,315389,20367,462.0


In [48]:
df.sort_values("price").head()

,product_id,route_id,price
8,48099,235,18.0
133,1418,1320,18.0
2,266892,76,18.0
107,13636,1130,18.0
1060,299790,9802,18.0


In [49]:
counts = []
prices = []
total = 0
for i,row in df.sort_values("price").iterrows():
    counts.append(len(counts)+1)
    total += row["price"]
    prices.append(total)

fig = go.Figure()
fig.add_trace(go.Scatter(x=counts, y=prices))
fig.update_layout(xaxis_title="#compounds", yaxis_title="Price EUR")

In [55]:
%%time
starting_recipes = hippo.Recipe.from_compounds(
    animal.compounds[list(df.sort_values("price").iloc[:200]["product_id"].unique())], 
    use_routes=True, 
    pick_cheapest=False,
)
starting_recipes

#compounds = 199

Output()

 Success  Found solution for compound=C1373!

 Success  Found solution for compound=C1399!

 Success  Found solution for compound=C1418!

...

 Success  Found solution for compound=C315074!

Solving recipe combinations...

Output()

CPU times: user 1.01 s, sys: 348 ms, total: 1.35 s
Wall time: 16 s


[Recipe({Ingredient × 276} --> {Ingredient × 1} --> {Ingredient × 199} via {R × 199}),
 Recipe({Ingredient × 277} --> {Ingredient × 199} via {R × 199})]

In [56]:
%%time
starting_recipe = starting_recipes[0]
starting_recipe.write_json("cycle_01/rgen/openbind_a71ev2a_c1_elabs_rgen_cheapest200_recipe.json")

 DISK  Writing 
/opt/xchem-fragalysis-2/maxwin/openbind-hippo/a71ev2a/cycle_01/rgen/openbind_a71ev2a_c1_elabs_rgen_cheapest200_reci
pe.json...

CPU times: user 3.26 s, sys: 8.01 s, total: 11.3 s
Wall time: 2min 31s


In [57]:
starting_recipe.price

€8131.20 EUR

### Do the recipe generation

In [25]:
gen = hippo.RandomRecipeGenerator(
    db=animal.db, 
    start_with=starting_recipe, 
    suppliers=['Enamine', 'Stock'],
    route_pool=route_pool,
    out_key="cycle_01/rgen/a71ev2a_c1_elabs"
)

DEBUG: RandomRecipeGenerator.__init__()

database = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/A71EV2A/A71EV2A.sqlite

max_lead_time = None

suppliers =
['Enamine', 'Stock']

out_key = cycle_01/rgen/a71ev2a_c1_elabs

 Warning  Will overwrite existing rgen data file: cycle_01/rgen/a71ev2a_c1_elabs_rgen.json!

#routes before pruning = 2079

#routes after pruning = 2079

Output()

 DISK  Writing cycle_01/rgen/a71ev2a_c1_elabs_rgen.json...

In [ ]:
recipe = gen.generate(balance_clusters=True, max_iter=1000)
recipe

In [ ]:
recipe.summary()

In [ ]:
recipe.reactants.price_df

In [ ]:
recipe = gen.generate(balance_clusters=False)
recipe

In [13]:
animal.elabs.poses.compounds.scaffolds

{C × 170}

## Export example series

In [6]:
elaborated_scaffolds = animal.compounds(tag="openbind_a71ev2a_c1_elaborated_scaffolds")

In [8]:
elaborated_scaffolds[0].elabs.poses.compounds

{C × 74}

In [11]:
elaborated_scaffolds[0]

C107 "ASAP-0031281-001"

In [12]:
elaborated_scaffolds[0].elabs.poses

{P × 203}

In [41]:
scaffold = animal.compounds["ASAP-0031281-001"]
poses = scaffold.poses + scaffold.elabs.poses
poses.to_fragalysis(
    f"cycle_01/syndirella/elabs/{scaffold.name}_elabs.sdf",
    method=f"{scaffold.name}_elabs",
    submitter_name="Max Winokan",
    submitter_institution="DLS",
    submitter_email="max.winokan@diamond.ac.uk",
)

DEBUG: 206 poses in set

DEBUG: 204 remaining after skipping null reference

DEBUG: 204 remaining after skipping null inspirations

#poses = 204

DEBUG: querying...

...

DEBUG: adding subsite column

out_path = 
/opt/xchem-fragalysis-2/maxwin/openbind-hippo/a71ev2a/cycle_01/syndirella/elabs/ASAP-0031281-001_elabs.sdf

[15:36:42] Molecule does not have explicit Hs. Consider calling AddHs()


 DISK  Writing 
/opt/xchem-fragalysis-2/maxwin/openbind-hippo/a71ev2a/cycle_01/syndirella/elabs/ASAP-0031281-001_elabs.sdf...

,HIPPO Pose ID,smiles,inchikey,alias,HIPPO Compound ID,ROMol,energy_score,distance_score,inspiration_score,Energy,...,fragmenstein_error,exports,GNINA pK,orig_smiles,tags,subsites,name,ref_mols,ref_pdb,_Name
0,45216,Cn1nc(C(=O)NCCc2cocn2)c2ccccc21,ONGQCMNCDUWDOR-UHFFFAOYSA-N,None,107,<rdkit.Chem.rdchem.Mol object at 0x7f2e85a85a80>,-27.234092,0.542984,None,73.960287,...,,[/opt/xchem-fragalysis-2/maxwin/BulkDock/OUTPU...,5.753,NaN,"fragmenstein_placed,BulkDock Fragalysis export...",1 - Active Site,P45216,"A2290a,A3436a",A2290a,P45216
1,56420,C[C@H](Cc1cocn1)NC(=O)c1nn(C)c2ccc(F)cc12,LLDZPHCJSJZSFN-UHFFFAOYSA-N,None,191887,<rdkit.Chem.rdchem.Mol object at 0x7f2e85a84d10>,13.542872,1.711796,None,99.979592,...,,[/opt/xchem-fragalysis-2/maxwin/openbind-hippo...,NaN,NaN,"fragmenstein_placed,openbind_a71ev2a_c1_fragme...",1 - Active Site,P56420,"A3510a,A4935a",A3510a,P56420
2,497883,Cc1ocnc1CCNC(=O)c1nn(C)c2ccc(F)cc12,SKWLCASPHPGKCF-UHFFFAOYSA-N,None,250891,<rdkit.Chem.rdchem.Mol object at 0x7f2e85a84130>,-21.210933,0.738298,None,NaN,...,NaN,[/opt/xchem-fragalysis-2/maxwin/openbind-hippo...,NaN,NaN,openbind_a71ev2a_c1_elabs_filtered,None,P497883,"A2290a,A3436a",A2290a,P497883
3,497884,Cc1nc(CCNC(=O)c2nn(C)c3ccc(F)cc23)co1,IIHQBSBEBQSMSZ-UHFFFAOYSA-N,None,289825,<rdkit.Chem.rdchem.Mol object at 0x7f2e85a84040>,-10.521861,0.533466,None,NaN,...,NaN,[/opt/xchem-fragalysis-2/maxwin/openbind-hippo...,NaN,NaN,openbind_a71ev2a_c1_elabs_filtered,None,P497884,"A2290a,A3436a",A2290a,P497884
4,497885,C[C@H](Cc1cocn1)NC(=O)c1nn(C)c2ccc(F)cc12,LLDZPHCJSJZSFN-SECBINFHSA-N,None,191887,<rdkit.Chem.rdchem.Mol object at 0x7f2e85a87b50>,-2.102867,0.710331,None,NaN,...,NaN,[/opt/xchem-fragalysis-2/maxwin/openbind-hippo...,NaN,NaN,openbind_a71ev2a_c1_elabs_filtered,None,P497885,"A2290a,A3436a",A2290a,P497885
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199,503519,COC(=O)C(Cc1cocn1)NC(=O)c1nn(CC2CNC(=O)O2)c2cc...,BMRKSVKHVNSEEK-UHFFFAOYSA-N,BMRKSVKHVNSEEK-UHFFFAOYSA-N-P183-P126-P183-601502,289894,<rdkit.Chem.rdchem.Mol object at 0x7f2e85862b60>,26.615833,1.319014,None,123.535359,...,,[/opt/xchem-fragalysis-2/maxwin/openbind-hippo...,NaN,COC(=O)C(Cc1cocn1)NC(=O)c1nn(CC2CNC(=O)O2)c2cc...,"fragmenstein_placed,a71ev2a_c1_elab_bulkdock_i...",None,BMRKSVKHVNSEEK-UHFFFAOYSA-N-P183-P126-P183-601502,"A2290a,A3436a",A0514a,BMRKSVKHVNSEEK-UHFFFAOYSA-N-P183-P126-P183-601502
200,503520,COC(=O)C(Cc1cocn1)NC(=O)c1nn(Cc2ccccc2)c2ccccc12,HTTSBEHOPRZHSA-UHFFFAOYSA-N,HTTSBEHOPRZHSA-UHFFFAOYSA-N-P126-P126-P183-601502,289895,<rdkit.Chem.rdchem.Mol object at 0x7f2e85862ca0>,-14.287370,1.945645,None,158.864274,...,,[/opt/xchem-fragalysis-2/maxwin/openbind-hippo...,NaN,COC(=O)C(Cc1cocn1)NC(=O)c1nn(Cc2ccccc2)c2ccccc12,"fragmenstein_placed,openbind_a71ev2a_c1_elabs_...",None,HTTSBEHOPRZHSA-UHFFFAOYSA-N-P126-P126-P183-601502,"A2290a,A3436a",A0514a,HTTSBEHOPRZHSA-UHFFFAOYSA-N-P126-P126-P183-601502
201,503521,COC(=O)C(Cc1cocn1)NC(=O)c1nn(Cc2ccccc2)c2ccccc12,HTTSBEHOPRZHSA-UHFFFAOYSA-N,HTTSBEHOPRZHSA-UHFFFAOYSA-N-P183-P126-P183-601502,289895,<rdkit.Chem.rdchem.Mol object at 0x7f2e85862fc0>,-9.979315,2.440847,None,163.172329,...,,[/opt/xchem-fragalysis-2/maxwin/openbind-hippo...,NaN,COC(=O)C(Cc1cocn1)NC(=O)c1nn(Cc2ccccc2)c2ccccc12,"fragmenstein_placed,a71ev2a_c1_elab_bulkdock_i...",None,HTTSBEHOPRZHSA-UHFFFAOYSA-N-P183-P126-P183-601502,"A2290a,A3436a",A0514a,HTTSBEHOPRZHSA-UHFFFAOYSA-N-P183-P126-P183-601502
202,503522,Cc1nc(CCNC(=O)c2nn(Cc3ccc(Cl)cc3Cl)c3ccccc23)c...,AKXUFMOJPOZWGY-UHFFFAOYSA-N,AKXUFMOJPOZWGY-UHFFFAOYSA-N-P126-P126-P183-601502,289896,<rdkit.Chem.rdchem.Mol object at 0x7f2e85862020>,21.514230,2.402301,None,138.449752,...,,[/opt/xchem-fragalysis-2/maxwin/openbind-hippo...,NaN,Cc1nc(CCNC(=O)c2nn(Cc3ccc(Cl)cc3Cl)c3ccccc23)c...,"fragmenstein_placed,a71ev2a_c1_elab_bulkdock_i...",None,AKXUFMOJPOZWGY-UHFFFAOYSA-N-P126-P126-P183-601502,"A2290a,A3436a",A0514a,AKXUFMOJPOZWGY-UHFFFAOYSA-N-P126-P126-P183-601502


In [38]:
df = poses.get_df()

DEBUG: querying...

Output()

 ERROR  None in smiles/inchikey column!

Output()

In [23]:
animal.compounds(smiles="Cn1nc(C(=O)NCCc2cocn2)c2ccccc21")

C107 "ASAP-0031281-001"

In [26]:
animal.P497883.smiles

'Cc1ocnc1CCNC(=O)c1nn(C)c2ccc(F)cc12'

In [39]:
df["smiles"].isna().any()

False

In [40]:
animal.C107.poses.names

['A2846a', 'ONGQCMNCDUWDOR-UHFFFAOYSA-N', 'ONGQCMNCDUWDOR-UHFFFAOYSA-N']